# Agent Architectures- Reason, Plan, Reflect

**Module 11 · Lesson 11.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Why Architecture Beats Model Choice
- ReAct - Reason & Act, Interleaved
- Plan-and-Execute - Plan First, Then Run
- ReWOO - Reasoning Without Observation
- Reflexion - Generate, Critique, Retry
- Routing & the Workflow Patterns

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com


## Same Task, Same Model, Four Very Different Bills

> 💡 **Info**
>
> Give an agent one job - “price the GenAI course with 18 percent GST, then compare it to Python” - and hold the model fixed. Solve it with **ReAct** and it makes one model call per step, reasoning as it goes. Solve it with **Plan-and-Execute** and it writes the whole plan in one call, then a cheap executor runs it. Solve it with **ReWOO** and it makes exactly *two* calls no matter how many tools. Solve it with **Reflexion** and it answers, grades itself, and retries until it is good. Same model, same tools, same answer - wildly different cost, latency, and failure modes. That choice is the **architecture**, and it is the highest-leverage decision you make after picking a model. This lesson builds all four from scratch, then hands you the ladder for choosing.

### 🎯 What you will be able to do after this lesson

- **Build** ReAct, Plan-and-Execute, ReWOO, and Reflexion as small loops over one shared tool set.

- **Compare** the patterns on LLM-call count, adaptivity, token cost, and failure mode - on the same task.

- **Operate** Anthropic’s workflow patterns (routing, prompt chaining, parallelization) and know which control-flow shape fits.

- **Evaluate** an architecture with the decision ladder (start simple, climb only when measured) and spot the “more agents = better” anti-pattern.

> 📦 **Info**
>
> ✅ Before you startThis builds on **11.1** (the observe-think-act loop; `stop_reason` drives it; reliability compounds as `p^N`) and **10.1–10.2** (a model picks a tool from its description + typed schema). You should be comfortable with the basic agent loop and with reading a short Python class. This lesson stays on **single-agent** architectures; multi-agent coordination is Lesson 11.5.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🍳 **Analogy**
>
> **Four ways to cook the same meal.** **ReAct** cooks by instinct: chop, taste, adjust, taste again - one step at a time. **Plan-and-Execute** reads the whole recipe, preps everything, then cooks in order. **ReWOO** writes the shopping list with blanks, sends one runner to fetch every ingredient at once, then assembles. **Reflexion** cooks, tastes the finished plate, says ‘too salty,’ and re-plates. Same kitchen, same ingredients - four different amounts of time, gas, and washing up. **Where it breaks down:** no single style is best; each wins on a different kind of dish.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A better model fixes a bad agent.”** Often the architecture is the bottleneck: the same model in a grounded loop (ReAct) beats a bigger model called blindly. Shape first, then scale the model.
> - **“More structure (or more agents) is always better.”** Every added call and hop compounds cost and failure. The winning move is usually the *simplest* pattern that clears the bar.

> 💡 **Info**
>
> ⚠️ Anti-patternPicking an architecture by fashion instead of task shape - reaching for a full autonomous agent when the inputs fall into three known categories (that is routing), or for a heavyweight plan when a single call would do. Match the control-flow shape to the task; start at the bottom of the ladder.

---

## 🎯 Concept 1: Why Architecture Beats Model Choice

### Why Architecture Beats Model Choice

The augmented LLM is the atom. HOW you wire the loop around it moves cost and reliability more than the model does.

Start with the atom every pattern is built from: the **augmented LLM** - a model plus the tools it can call (and, later, retrieval and memory). By itself it answers in one shot. An **architecture** is how you wire calls to that atom: in a sequence, a branch, a loop, a plan. The claim of this whole lesson is that for real tasks, that wiring moves **cost, latency, and reliability** more than swapping the model does - a mid-tier model in the right architecture routinely beats a frontier model called blindly. So the engineering question is not just “which model?” but “which *shape*?”

> 🏁 **Analogy**
>
> It’s the **engine versus the chassis**. The model is the engine; the architecture is the chassis and gearbox around it. Drop a great engine into a go-kart and it still loses to a decent engine in a proper race car. On the track (production), the chassis - how the power is delivered - decides the lap time as much as raw horsepower.

Team A upgrades to a bigger model but keeps a single blind call. Team B keeps the model but wraps it in a ReAct loop with a calculator. Who gets ‘14999 with 18% GST’ right?

**📝 `01_augmented_llm.py`** - *runnable*

In [ ]:
# THE AUGMENTED LLM - the atom every architecture composes: a model + tools (+ retrieval, memory).
# HOW you wire the calls around this atom - the architecture - moves cost and reliability
# more than swapping the model does. We script the model so the trace is deterministic.
KB = {"genai": "GenAI course: 14999 INR", "python": "Python course: 9999 INR"}
def search(q):  return KB.get(q.lower().split()[0], "not found")   # a retrieval tool
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)  # an action tool
TOOLS = {"search": search, "calc": calc}

# A direct call would GUESS the price and the GST. The augmented LLM can look it up and compute it.
print("augmented LLM = model +", list(TOOLS))
print("search('genai') ->", search("genai"))
print("calc('14999*1.18') ->", calc("14999*1.18"))
print("same model, same tools - the architecture decides how many calls this takes")

# Output:
# augmented LLM = model + ['search', 'calc']
# search('genai') -> GenAI course: 14999 INR
# calc('14999*1.18') -> 17698.82
# same model, same tools - the architecture decides how many calls this takes

- The augmented LLM = the model PLUS the tools it can call (`search`, `calc` here).
- A direct call would *guess* the price and the GST; the tools let it look up and compute.
- The same atom can be wired many ways - the next six steps are those wirings.
- Architecture is the count and shape of the calls, not the model behind them.

#### 💡 What just happened

⚡ What just happened? The **augmented LLM** (model + tools) is the building block; the **architecture** is how you wire calls around it. Tradeoff / when it matters: on a hard task the shape moves cost and reliability more than the model tier does - so choose the shape deliberately. The next four steps are the four canonical shapes.

- Toggle the model tier vs the architecture and watch cost/reliability/latency bars
- See the same task get cheaper or more reliable from wiring, not the model

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: ReAct - Reason & Act, Interleaved

### ReAct - Reason & Act, Interleaved

One LLM call per step; the path is decided as it goes. Adaptive, transparent, but token-hungry.

**ReAct** (Reason + Act) is the adaptive shape: the agent interleaves a **Thought** (“this needs arithmetic”), an **Action** (call a tool), and an **Observation** (the result), and repeats - deciding the next step only after seeing the last one. This is the loop from 11.1, now named as an architecture. Its strength is that every step is *grounded* in a real observation, so it adapts when the path is unknown. Its cost: **one model call per step**, so tokens and latency grow with the number of steps, and it can wander into loops (which is why you cap iterations). ReAct is the default single-agent pattern - and the one every framework ships prebuilt.

> 🍳 **Analogy**
>
> It’s a **cook tasting as they go**: add salt, taste, adjust, taste again - one decision at a time, always reacting to the last taste. Flexible and reliable, but slow if the dish needs many tastes; each ‘taste’ is another model call.

A ReAct agent needs the GenAI price and then the GST. How many LLM calls does that take?

**📝 `02_react.py`** - *runnable*

In [ ]:
# ReAct - reason & act, INTERLEAVED. One LLM call per step; the path is decided as it goes.
KB = {"genai": "GenAI course: 14999 INR", "python": "Python course: 9999 INR"}
def search(q):  return KB.get(q.lower().split()[0], "not found")
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)
TOOLS = {"search": search, "calc": calc}
calls = {"n": 0}

def think(task, scratch):                          # SCRIPTED stand-in for one LLM call
    calls["n"] += 1
    used = [s for s in scratch if s.startswith("Act")]
    if not any("search" in u for u in used):
        return "Thought: I need the GenAI price first.", ("search", "genai")
    if not any("calc" in u for u in used):
        obs = used[0].split("->")[-1]                       # "GenAI course: 14999 INR"
        price = "".join(c for c in obs if c.isdigit())      # -> "14999"
        return "Thought: now add 18% GST.", ("calc", f"{price}*1.18")
    return "Thought: I have the answer.", ("finish", scratch[-1].split("->")[-1].strip())

def react(task, max_steps=5):
    scratch = []
    for step in range(max_steps):
        thought, (tool, arg) = think(task, scratch)     # REASON
        if tool == "finish":
            n_steps = len([s for s in scratch if s.startswith("Act")])
            print(f"  {thought}\n  Answer: {arg} INR ({n_steps} tool steps, {calls['n']} LLM calls)")
            return
        obs = TOOLS[tool](arg)                           # ACT
        scratch.append(thought)
        scratch.append(f"Act {tool}({arg}) -> {obs}")    # OBSERVE (fed back next step)
        print(f"  step {step+1}: {thought}  |  {tool}({arg}) -> {obs}")

react("What is the GenAI course price with 18% GST?")

# Output:
#   step 1: Thought: I need the GenAI price first.  |  search(genai) -> GenAI course: 14999 INR
#   step 2: Thought: now add 18% GST.  |  calc(14999*1.18) -> 17698.82
#   Thought: I have the answer.
#   Answer: 17698.82 INR (2 tool steps, 3 LLM calls)

- Each step is a Thought (reason) then an Action (a tool call) then the Observation.
- The scripted `think` stands in for one model call; a real model decides this.
- It grounds step 2 (the GST calc) on step 1’s observed price - that is the adaptivity.
- Two tool steps here take **3 LLM calls**; on a 6-step task it would take 7. Cost scales with N.

**📝 `02b_react_real.py`** - *Anthropic*

In [ ]:
# ReAct FOR REAL - the same loop with the Anthropic Messages API (claude-sonnet-4-6).
import anthropic
client = anthropic.Anthropic()                       # reads ANTHROPIC_API_KEY
TOOLS = [{"name": "calc", "description": "Do arithmetic, e.g. 14999*1.18.",
          "input_schema": {"type": "object", "properties": {"expr": {"type": "string"}},
                           "required": ["expr"]}}]
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)

messages = [{"role": "user", "content": "What is 14999 with 18% GST?"}]
max_steps = 5
for step in range(max_steps):                        # cap iterations - 11.1's reliability rule
    resp = client.messages.create(model="claude-sonnet-4-6", max_tokens=1024,
                                  tools=TOOLS, messages=messages)
    if resp.stop_reason != "tool_use":               # not tool_use -> the model reasoned to an answer
        print(resp.content[0].text); break
    messages.append({"role": "assistant", "content": resp.content})
    results = [{"type": "tool_result", "tool_use_id": b.id, "content": str(calc(**b.input))}
               for b in resp.content if b.type == "tool_use"]     # ACT
    messages.append({"role": "user", "content": results})         # OBSERVE

# Output: (illustrative minimal example - needs `pip install anthropic` + a key; in production add
#   a request timeout= and try/except.) The model REASONS in text between tool calls and ACTS via
#   tool_use - that interleaving IS ReAct. LangChain 1.0 gives you this for free:
#   from langchain.agents import create_agent; agent = create_agent(model, tools)  # runs the loop.

- This is 11.1’s `stop_reason` loop: `tool_use` → run the tool, append a `tool_result`, repeat.
- The model’s reasoning lives in the text between tool calls; the tool calls are the actions - that is ReAct.
- `max_steps` caps it, exactly the reliability rule from 11.1.
- LangChain 1.0’s `create_agent` (and LangGraph’s prebuilt ReAct agent) run this loop for you.

#### 💡 What just happened

⚡ What just happened? ReAct = interleave reason, act, observe, one model call per step. Tradeoff: maximum adaptivity, but cost and latency grow with the step count and it can loop - so it is the default, not the cheapest. The next three patterns cut the call count in different ways.

- Step the Thought -> Action -> Observation scratchpad on a task
- Watch the per-step LLM-call counter climb with N

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Plan-and-Execute - Plan First, Then Run

### Plan-and-Execute - Plan First, Then Run

A capable planner writes the whole plan in one call; a cheap executor runs it. Fewer calls, but a brittle plan.

ReAct decides one step at a time; **Plan-and-Execute** decides them all at once. A capable model writes the **full plan** up front in a single call, then a *cheaper* executor runs each step without re-reasoning - far fewer model calls on the hot path, and a plan you can inspect before anything runs. The catch is the **brittle plan**: the planner commits before it has seen a single tool result, so if step 2 returns something it did not anticipate, step 3 is already written and wrong. The fix is to **replan** - call the planner again with what you have learned - which trades some of the call-count savings back. Best when the task decomposes cleanly and the world is predictable.

> 🗺️ **Analogy**
>
> It’s a **GPS route**. It computes the entire route before you pull out of the driveway (one planning call), then you just follow the turns (cheap execution). But if a road is closed - something the plan did not foresee - it has to **reroute** (replan) from where you are.

A Plan-and-Execute agent’s step 2 returns something the plan did not expect. What happens to step 3?

**📝 `03_plan_execute.py`** - *runnable*

In [ ]:
# PLAN-AND-EXECUTE - a capable planner writes the whole plan FIRST; a cheap executor runs it.
KB = {"genai": "GenAI course: 14999 INR", "python": "Python course: 9999 INR"}
def search(q):  return KB.get(q.lower().split()[0], "not found")
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)
TOOLS = {"search": search, "calc": calc}

def plan(task):                                   # ONE planner call, up front
    return [{"tool": "search", "arg": "genai"},
            {"tool": "search", "arg": "python"},
            {"tool": "calc",   "arg": "14999*1.18"},
            {"tool": "calc",   "arg": "9999*1.18"},
            {"tool": "answer", "arg": "compare the two GST-inclusive prices"}]

def execute(task):
    steps = plan(task)
    print(f"  planned {len(steps)} steps up front (1 planner call), now executing:")
    results = []
    for i, s in enumerate(steps, 1):
        if s["tool"] == "answer":
            print(f"    {i}. ANSWER: GenAI {results[2]} vs Python {results[3]} INR (with GST)")
            return
        out = TOOLS[s["tool"]](s["arg"])
        results.append(out)
        print(f"    {i}. {s['tool']}({s['arg']}) -> {out}")

execute("Compare GenAI and Python prices, both with 18% GST")

# If a step's result surprises the plan, we REPLAN (call the planner again). Demo:
def execute_with_replan(bad_step=2):
    steps = plan("x")
    for i, s in enumerate(steps, 1):
        if i == bad_step:
            print(f"  step {i} returned the unexpected -> REPLAN (1 extra planner call)")
            return
        print(f"  step {i}: {s['tool']}({s['arg']}) ok")
print("\n  brittle-plan demo:")
execute_with_replan()

# Output:
#   planned 5 steps up front (1 planner call), now executing:
#     1. search(genai) -> GenAI course: 14999 INR
#     2. search(python) -> Python course: 9999 INR
#     3. calc(14999*1.18) -> 17698.82
#     4. calc(9999*1.18) -> 11798.82
#     5. ANSWER: GenAI 17698.82 vs Python 11798.82 INR (with GST)
#   brittle-plan demo:
#   step 1: search(genai) ok
#   step 2 returned the unexpected -> REPLAN (1 extra planner call)

- `plan()` is ONE planner call that lays out all five steps before any tool runs.
- The executor then runs them cheaply, with no per-step reasoning - fewer model calls than ReAct.
- The plan is a plain list you can read and audit before executing - a real benefit.
- The brittle-plan demo shows the fix: a surprising result triggers a **replan**.

#### 💡 What just happened

⚡ What just happened? Plan-and-Execute splits planning (one capable call) from execution (cheap), cutting hot-path calls. Tradeoff: the plan is committed before any observation, so it is brittle - you add replanning to recover. Use it when the task decomposes cleanly; reach for ReAct when the path keeps surprising you.

- Build the whole plan up front, then run the cheap executor
- Flip a step to 'fails' and watch the replan branch fire

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: ReWOO - Reasoning Without Observation

### ReWOO - Reasoning Without Observation

Plan with placeholders, batch the tools, solve once. Exactly two LLM calls; the token-efficient corner of the plan family.

**ReWOO** (Reasoning WithOut Observation) is Plan-and-Execute taken to its efficient extreme. The planner writes every step at once but leaves **placeholders** (`#E1`, `#E2`) where a tool result will go, *without* pausing to observe each one. All the tools then run in a **batch** (independent ones in parallel), and a single **solve** call integrates the evidence into the answer. The result: **exactly two model calls** - plan and solve - no matter how many tools, which is why ReWOO is roughly **five times more token-efficient** than ReAct on multi-tool tasks. The price you pay is the same as Plan-and-Execute, only sharper: **zero mid-run adaptation**. If a tool returns the unexpected, the placeholder chain breaks and there is no observation step to catch it.

> 🛒 **Analogy**
>
> It’s a **shopping list with blanks**. Instead of walking to the shop for each item and back (ReAct), you write the whole list - ‘price of A is ___, price of B is ___’ - hand it to one runner who fetches everything in a single trip, then you total it up. Two thoughts (write the list, add it up), one trip.

A task needs two independent lookups. ReAct makes ~3 calls; how many does ReWOO make, and why?

**📝 `04_rewoo.py`** - *runnable*

In [ ]:
# ReWOO - Reasoning WithOut Observation. Plan ALL steps with #E placeholders, run tools in a
# batch, then solve once. Exactly TWO LLM calls (plan + solve), no matter how many tools.
KB = {"genai": "GenAI course: 14999 INR", "python": "Python course: 9999 INR"}
def search(q):  return KB.get(q.lower().split()[0], "not found")
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)
TOOLS = {"search": search, "calc": calc}
calls = {"n": 0}

def planner():                                    # LLM call #1: plan with placeholders
    calls["n"] += 1
    return [("#E1", "calc", "14999*1.18"),        # steps do not wait on each other
            ("#E2", "calc", "9999*1.18")]

def solver(evidence):                             # LLM call #2: integrate the evidence
    calls["n"] += 1
    return f"GenAI {evidence['#E1']} vs Python {evidence['#E2']} INR, both incl. 18% GST"

def rewoo():
    plan = planner()
    evidence = {var: TOOLS[tool](arg) for var, tool, arg in plan}   # BATCH exec, parallelizable
    for var, tool, arg in plan:
        print(f"  {var} = {tool}({arg}) -> {evidence[var]}")
    print("  solve:", solver(evidence))
    print(f"  total LLM calls: {calls['n']}  (ReAct would need one PER step, growing with N)")

rewoo()

# Output:
#   #E1 = calc(14999*1.18) -> 17698.82
#   #E2 = calc(9999*1.18) -> 11798.82
#   solve: GenAI 17698.82 vs Python 11798.82 INR, both incl. 18% GST
#   total LLM calls: 2  (ReAct would need one PER step, growing with N)

- The planner emits steps with placeholders (`#E1`, `#E2`) and does not wait on results.
- The tools run in a batch (here two independent `calc`s, parallelizable) filling the placeholders.
- One `solve` call turns the evidence into the final answer.
- Total: **2 LLM calls** vs ReAct’s one-per-step - the source of ReWOO’s token savings.

#### 💡 What just happened

⚡ What just happened? ReWOO = plan with placeholders → batch the tools → solve once = exactly two model calls, about 5x fewer tokens than ReAct on multi-tool work. Tradeoff: it cannot adapt mid-run - if a tool surprises the plan, the chain breaks. Use it when the sub-steps are known and independent; avoid it when the path is uncertain.

- Plan with #E1/#E2 placeholders, run the tools in a batch, then solve
- Compare the 2-call ReWOO counter to ReAct's growing one; watch the token bar shrink

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Reflexion - Generate, Critique, Retry

### Reflexion - Generate, Critique, Retry

Answer, grade yourself, keep the critique in memory, and try again. Quality through iteration - at a price.

The patterns so far decide *how* to reach an answer. **Reflexion** adds a quality loop *around* the answer: the agent generates a full attempt, an **evaluator** scores it and lists flaws, and if it falls short the agent writes a **reflection** (“I forgot the GST”), stores it in memory, and retries with that critique in context. It genuinely learns within a session - on coding benchmarks, self-reflection lifts pass rates markedly (HumanEval pass@1 rose from about **80% to 91%** in the original work). This is Anthropic’s *evaluator-optimizer* workflow in agent form. The cost is real: each retry is a **full run**, so Reflexion is the most expensive pattern, and a self-evaluator can reinforce its own blind spots. It earns its keep only when there is a **checkable quality bar** to iterate against.

> 📝 **Analogy**
>
> It’s a **chef who plates, tastes, and re-plates**. First plate: ‘too salty.’ They note *why*, then cook again with less salt. The tasting note carried into the next attempt is the reflection; the dish improves, but you paid to cook it twice.

Reflexion’s attempt 1 scores 6/10 (‘forgot the GST’). What makes attempt 2 better?

**📝 `05_reflexion.py`** - *runnable*

In [ ]:
# REFLEXION - generate, self-critique, RETRY with the critique in memory. Quality via iteration.
def generate(task, reflections):                  # one full attempt (an LLM call)
    if not reflections:
        return "GenAI course is 14999 INR."                       # attempt 1: forgot the GST
    return "GenAI course is 14999 INR; with 18% GST that is 17698.82 INR."  # attempt 2: fixed

def evaluate(answer):                             # the evaluator scores + lists flaws
    if "GST" not in answer and "17698" not in answer:
        return {"score": 6, "flaws": ["did not apply the 18% GST the task asked for"]}
    return {"score": 9, "flaws": []}

def reflexion(task, max_attempts=3, bar=8):
    reflections = []
    for attempt in range(1, max_attempts + 1):
        answer = generate(task, reflections)
        verdict = evaluate(answer)
        print(f"  attempt {attempt}: score {verdict['score']}/10  ->  {answer}")
        if verdict["score"] >= bar:
            print(f"  accepted after {attempt} attempt(s)"); return
        reflections.append(f"Reflection: {verdict['flaws'][0]}")   # stored, fed to next attempt
        print(f"    reflect: {reflections[-1]}")

reflexion("Give the GenAI course price WITH 18% GST.")

# Output:
#   attempt 1: score 6/10  ->  GenAI course is 14999 INR.
#     reflect: Reflection: did not apply the 18% GST the task asked for
#   attempt 2: score 9/10  ->  GenAI course is 14999 INR; with 18% GST that is 17698.82 INR.
#   accepted after 2 attempt(s)

- Attempt 1 answers without the GST and the evaluator scores it 6/10 with a specific flaw.
- The `reflection` (‘did not apply the 18% GST’) is stored and fed into the next attempt.
- Attempt 2 applies the GST, scores 9/10, and is accepted - quality through iteration.
- Each attempt is a full run, so cost grows with the number of retries.

#### 💡 What just happened

⚡ What just happened? Reflexion = generate → self-critique → retry with the critique in memory; it lifts quality where there is a checkable bar (HumanEval ~80→91%). Tradeoff: it is the priciest pattern (every retry is a full run) and can reinforce its own blind spots. Reach for it on precision-critical work with a clear quality signal, not on everything.

- Run an attempt, read the evaluator score, add a reflection, retry
- Watch the score climb 6 -> 9 and the cost bar grow with each retry

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Routing & the Workflow Patterns

### Routing & the Workflow Patterns

Classify, then send to a specialized handler. Plus chaining and parallelization: the fixed-code shapes.

The four patterns above are *agent* shapes (the model decides the path). Anthropic’s **workflow** patterns are the opposite: *you* fix the control flow in code, and they are often the highest-ROI choice. The standout is **routing**: a small, cheap classifier reads the input, picks a category, and hands off to a **specialized handler** - each with its own prompt, tools, and even model (billing on a cheap model with account tools; technical on a stronger model with docs RAG). Because the handlers are specialized, quality goes up and cost goes down; the one thing that can sink it is **classification accuracy**, so you invest in the router. Its cousins: **prompt chaining** (a fixed sequence of calls with gates between - trade latency for accuracy) and **parallelization** (run independent subtasks at once, or vote the same task N times for confidence). These three are the workflow patterns you wire here; Anthropic’s taxonomy names **five** in total, and you have already met the other two - **evaluator-optimizer** is Reflexion’s workflow cousin (Step 5), and **orchestrator-workers** sits at the top of the ladder just before multi-agent (Step 7). All are composable building blocks.

> 📧 **Analogy**
>
> Routing is a **mailroom sorter**: it reads each envelope’s address and drops it in the right bin - billing, technical, returns - where a specialist handles it. The sorter is fast and cheap; the whole system only fails if the sorter misreads the address (the classifier is wrong).

Support queries fall into billing, technical, and returns - three known categories. Best pattern?

**📝 `06_routing.py`** - *runnable*

In [ ]:
# ROUTING - a cheap classifier sends each query to a SPECIALIZED handler (own model/prompt/tools).
# The highest-ROI workflow pattern when inputs fall into known categories.
def classify(query):                              # the router (a small, cheap call)
    q = query.lower()
    if any(w in q for w in ("refund", "charge", "invoice", "bill")): return "billing"
    if any(w in q for w in ("error", "crash", "bug", "install")):    return "technical"
    if any(w in q for w in ("return", "exchange", "replace")):       return "returns"
    return "general"

HANDLERS = {                                      # each handler is tuned for its domain
    "billing":   lambda q: "[billing/Haiku + account tools] checking your invoice...",
    "technical": lambda q: "[technical/Sonnet + docs RAG] reproducing the error...",
    "returns":   lambda q: "[returns/order tools] starting a return...",
    "general":   lambda q: "[general] routing to a human.",
}
def route(query):
    cat = classify(query)
    print(f"  {query!r:42s} -> {cat:9s} -> {HANDLERS[cat](query)}")

for q in ["I was charged twice this month",
          "the app crashes on install",
          "I want to return my order",
          "what are your opening hours"]:
    route(q)

# Output:
#   'I was charged twice this month'           -> billing   -> [billing/Haiku + account tools] checking your invoice...
#   'the app crashes on install'               -> technical -> [technical/Sonnet + docs RAG] reproducing the error...
#   'I want to return my order'                -> returns   -> [returns/order tools] starting a return...
#   'what are your opening hours'              -> general   -> [general] routing to a human.

- `classify` is the cheap router; it maps each query to billing / technical / returns / general.
- Each handler is specialized - its own model, prompt, and tools (shown in brackets).
- A misroute sends the query to the wrong specialist, so the classifier’s accuracy is the bottleneck.
- Chaining and parallelization are the other two fixed-code shapes named in the prose.

#### 💡 What just happened

⚡ What just happened? Routing = a cheap classifier → specialized handlers; it is the highest-ROI workflow when inputs fall into known categories. Tradeoff: the router’s accuracy caps the whole system, so invest there. Chaining and parallelization round out the fixed-code shapes; all three compose with the agent patterns above.

- Send a query and watch the classifier fan it to billing / technical / returns
- Toggle chaining vs parallelization to see the other workflow shapes

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Choosing an Architecture - the Decision Ladder

### Choosing an Architecture - the Decision Ladder

Start at the simplest rung; climb only when a measurement forces it. The anti-pattern is climbing for fashion.

With the shapes in hand, the skill is **choosing**. Anthropic’s guidance is a ladder: start at the *simplest* rung and climb only when a measurement shows the added complexity pays for itself. The rungs, roughly: a **single call** for a one-shot task → **prompt chaining** for fixed steps → **routing** for known categories → **parallelization** for independent subtasks → **Plan/ReWOO** when the task decomposes cleanly → **ReAct** when the path must be decided at runtime → **Reflexion** when there is a quality bar to iterate against → **orchestrator-workers** when subtasks are unknown until runtime - and only then **multi-agent** (Lesson 11.5). The dominant anti-pattern is climbing for fashion: reaching for a heavyweight pattern (or more agents) when a lower rung clears the bar. Every added call and hop compounds cost and failure - so the burden of proof is on the more complex option.

> 🧩 **Analogy**
>
> It’s **choosing a vehicle for the trip**. Across the room, walk (a single call). Across town, a bike or car (a workflow). Across the country hauling freight, a truck (an agent, or multi-agent). Bringing the truck to cross the room is not ‘more powerful’ - it is just slower, costlier, and more likely to break down.

A nightly job runs the same five fixed steps every time, no runtime decisions. Which rung?

**📝 `07_decision.py`** - *runnable*

In [ ]:
# THE DECISION LADDER - start at the simplest rung; climb ONLY when the task forces it.
def choose(single_shot, fixed_steps, known_categories, independent_parallel,
           path_unknown, decomposes_cleanly, checkable_quality):
    if single_shot:            return "just call the API",     "one shot, no control flow needed"
    if fixed_steps:            return "prompt chaining",       "fixed sub-steps with gates between"
    if known_categories:       return "routing",               "classify -> specialized handler"
    if independent_parallel:   return "parallelization",       "independent subtasks, or vote"
    if decomposes_cleanly:     return "Plan-and-Execute/ReWOO","plan up front, cheap execution"
    if path_unknown:           return "ReAct (agent)",         "decide the next step at runtime"
    if checkable_quality:      return "Reflexion",             "a clear quality bar to iterate against"
    return "orchestrator-workers -> multi-agent (11.5)", "dynamic subtasks; go up only if measured"

cases = [
    ("translate one line",       dict(single_shot=1, fixed_steps=0, known_categories=0, independent_parallel=0, path_unknown=0, decomposes_cleanly=0, checkable_quality=0)),
    ("summarize then translate", dict(single_shot=0, fixed_steps=1, known_categories=0, independent_parallel=0, path_unknown=0, decomposes_cleanly=0, checkable_quality=0)),
    ("support triage (3 cats)",  dict(single_shot=0, fixed_steps=0, known_categories=1, independent_parallel=0, path_unknown=0, decomposes_cleanly=0, checkable_quality=0)),
    ("draft code to a test",     dict(single_shot=0, fixed_steps=0, known_categories=0, independent_parallel=0, path_unknown=0, decomposes_cleanly=0, checkable_quality=1)),
    ("open-ended web research",  dict(single_shot=0, fixed_steps=0, known_categories=0, independent_parallel=0, path_unknown=1, decomposes_cleanly=0, checkable_quality=0)),
]
for name, flags in cases:
    pick, why = choose(**flags)
    print(f"  {name:26s} -> {pick:34s} ({why})")

# Output:
#   translate one line         -> just call the API                  (one shot, no control flow needed)
#   summarize then translate   -> prompt chaining                    (fixed sub-steps with gates between)
#   support triage (3 cats)    -> routing                            (classify -> specialized handler)
#   draft code to a test       -> Reflexion                          (a clear quality bar to iterate against)
#   open-ended web research    -> ReAct (agent)                      (decide the next step at runtime)

- `choose` encodes the ladder: it returns the *lowest* rung whose condition the task meets.
- One-shot → API call; fixed steps → chaining; categories → routing; checkable quality → Reflexion.
- Open-ended, decide-at-runtime tasks are the ones that actually justify ReAct or an agent.
- The last rung (orchestrator-workers → multi-agent) is Lesson 11.5 - climb there only when measured.

#### 💡 What just happened

⚡ What just happened? The decision ladder: start at the simplest rung and climb only when a metric justifies it. Tradeoff / anti-pattern: every rung up adds cost and failure surface, so “more structure” and “more agents” are the default mistakes, not the default answers. Match the control-flow shape to the task shape.

- Slide a task up and down the rungs from single-call to multi-agent
- Watch each task settle on its lowest fitting rung; flag the fashion anti-pattern

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> The **augmented LLM** (model + tools) is the atom, and the **architecture** - how you wire calls around it - moves cost and reliability more than the model does (Step 1). **ReAct** reasons and acts one grounded step at a time: adaptive, but one call per step (Step 2). **Plan-and-Execute** plans in one call and executes cheaply, at the cost of a brittle plan you must replan (Step 3). **ReWOO** pushes that to exactly two calls with placeholders and batched tools - efficient, but zero mid-run adaptation (Step 4). **Reflexion** wraps a self-critique-and-retry loop around the answer: higher quality where there is a checkable bar, at the highest cost (Step 5). Anthropic’s **workflow** patterns - routing, chaining, parallelization - fix the control flow in code and are often the highest-ROI choice (Step 6). And the **decision ladder** tells you to start simple and climb only when measured (Step 7). Same model, many shapes - you now pick the shape on purpose.

The four reasoning patterns, side by side on the axes from Objective 2 (call counts are the shape of the pattern, not a benchmark):

| Pattern | LLM calls (hot path) | Adaptivity | Relative token cost | Dominant failure mode |
|---|---|---|---|---|
| ReAct | one per step (~N) | high - grounds each step | high | myopic; can wander or loop |
| Plan-and-Execute | 1–3 (plan, execute, replan) | low - per plan | medium | brittle plan; a surprise cascades |
| ReWOO | exactly 2 (plan + solve) | none mid-run | lowest (~5x under ReAct) | placeholder chain breaks on a surprise |
| Reflexion | N attempts + eval each | across attempts | highest | costly; self-critique blind spots |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now name and build the single-agent architectures. Next, Module 11 goes deeper: **LangGraph** gives these shapes a real state machine (checkpointing, time-travel, human-in-the-loop) in Lesson 11.3; **memory systems** in Lesson 11.4 replace the toy history with episodic and long-term stores; **multi-agent** coordination - supervisor, swarm, handoffs - is Lesson 11.5; and **evaluation and security** for agents is Lesson 11.11. The reference is Anthropic’s [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Agent Architectures- Reason, Plan, Reflect**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.2.html` - regenerate this notebook whenever the source HTML is updated.*
